# 真实 API：CoVe 验证链修正

CoVe 是唯一支持 API 后端的缓解方法：生成基线回答，规划验证问题，独立作答，最后综合生成修正回答。四阶段全部走真实 API。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai]"`并配置 `OPENAI_API_KEY`；`OPENAI_BASE_URL` 可指向任意 OpenAI-compatible 服务（vLLM/Ollama）。会产生多次真实调用费用。

代码单元没有预执行。填写本地模型路径或 API 凭据后，从上到下运行；不要将 API key 保存到 notebook。缓解会改变模型输出，务必同时阅读基线与缓解后的回答。

## 参数说明

- `OPENAI_MODEL: str`：API 模型名。
- `max_questions: int = 6`：验证问题数。
- `mode: str = 'factored'`：factored（独立作答）或 joint（联合作答）。
- `temperature: float = 0.0`：各阶段采样温度。

## 步骤 1：配置真实 API

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
OPENAI_MODEL = "gpt-4.1-mini"
OPENAI_BACKEND = "openai"                     # or "openai-compatible" for local vLLM/Ollama servers
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key        # the Mitigator reads the key from this environment variable
API_BASE = os.environ.get("OPENAI_BASE_URL")  # optional: point at an OpenAI-compatible server

## 步骤 2：构建 Mitigator

In [ ]:
from hallucination.mitigation import Mitigator
mitigator = Mitigator(
    method="cove",
    backend=OPENAI_BACKEND,
    model=OPENAI_MODEL,
    api_base=API_BASE,
    api_key_env="OPENAI_API_KEY",
    max_questions=5,
    mode="factored",
    temperature=0.0,
    baseline=True,
)

## 步骤 3：交互运行并读取

In [ ]:
prompt = input("Prompt：").strip()
report = mitigator.mitigate(prompt, max_new_tokens=256)
print("baseline :", report.baseline)
print("revised  :", report.response)
print("warnings:", report.warnings)

## 结果解读与限制

CoVe 用同一模型自问自答，是自验证而非外部核查，验证答案本身可能也是幻觉；真实冒烟中还出现过回答语言漂移（中文提问英文作答），生产使用必须固定输出语言并抽查验证答案。